# vLLM Scheduler Optimization

## Objective

The goal of this notebook is to modify the vLLM scheduler policy and evaluate whether the new policy improves latency for mixed inference workloads without significantly degrading decode efficiency or overall throughput.

Previous source-level tracing and baseline experiments showed that long-prefill requests can interfere with short requests that arrive shortly afterward.

Under the mixed-arrival workload:

Long request starts  
↓  
Long prefill begins  
↓  
Short request arrives 50 ms later  
↓  
Short request waits for a scheduler opportunity  
↓  
First token is generated  

The baseline experiments showed that reducing the scheduler token budget from 8192 to 512 improved short-request TTFT while leaving TPOT almost unchanged.

This suggests that the main performance opportunity is in prefill scheduling rather than decode execution.

## Baseline Observation

With automatic prefix caching disabled, the mixed-arrival streaming benchmark produced the following steady-state results.

### Default Scheduler

`max_num_batched_tokens = 8192`

Short request:

- mean TTFT ≈ 1.095 s
- p95 TTFT ≈ 1.143 s
- mean TPOT ≈ 19.60 ms/token
- mean end-to-end latency ≈ 1.702 s

### Controlled Scheduler

`max_num_batched_tokens = 512`

Short request:

- mean TTFT ≈ 1.015 s
- p95 TTFT ≈ 1.022 s
- mean TPOT ≈ 19.46 ms/token
- mean end-to-end latency ≈ 1.618 s

The smaller token budget reduced short-request mean TTFT by approximately 7% and p95 TTFT by approximately 11%, while TPOT remained nearly unchanged.

## Optimization Motivation

A fixed global token budget affects all workloads equally.

However, different inference phases have different scheduling characteristics:

- long prefills can consume hundreds or thousands of tokens in one scheduler iteration
- decode requests usually require only one token per iteration
- newly arrived short requests may experience large TTFT if a long prefill dominates the available token budget

The optimization goal is therefore to make prefill scheduling more responsive to competing decode or short requests.

Instead of only changing the global `max_num_batched_tokens`, this notebook will explore a scheduler-level policy that explicitly limits or adapts the amount of long-prefill work scheduled per iteration.

## Optimization Direction

The initial policy will focus on prefill-aware scheduling.

Conceptually:

Scheduler step  
↓  
Identify active decode work  
↓  
Identify long-prefill work  
↓  
Protect capacity for latency-sensitive requests  
↓  
Limit long-prefill chunk size when necessary  
↓  
Use remaining budget for other requests  

The optimization should preserve the existing vLLM execution model while changing only the scheduling decision.

## Evaluation Criteria

The modified scheduler will be evaluated using the same workloads and metrics as the baseline.

Primary metrics:

- short-request mean TTFT
- short-request p95 TTFT
- long-request TTFT
- TPOT
- end-to-end latency

Secondary metrics:

- output throughput
- scheduler token utilization
- number and size of prefill chunks
- KV-cache pressure
- preemption behavior

## Experimental Principle

Only the scheduler policy should change.

The following factors should remain consistent across baseline and optimized experiments:

- model
- GPU
- prompt lengths
- output lengths
- arrival delay
- sampling parameters
- prefix-caching setting
- maximum model length
- GPU memory utilization

## Workflow

Baseline scheduler  
↓  
Measure behavior  
↓  
Modify scheduler policy  
↓  
Instrument scheduling decisions  
↓  
Run identical workloads  
↓  
Compare TTFT, TPOT, and latency  
↓  
Evaluate trade-offs  

The optimization will be considered useful only if latency improvements can be explained by scheduler behavior and do not introduce unacceptable regressions in throughput or decode performance.

### 0. Confirm the GPU

In [1]:
!nvidia-smi

Mon Sep 21 02:21:44 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip uninstall -y torchaudio
!pip install -U torchaudio==2.11.0+cu130 \
  --index-url https://download.pytorch.org/whl/cu130

Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128
Looking in indexes: https://download.pytorch.org/whl/cu130
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 39.2 MB/s eta 0:00:00


In [3]:
%pip install -q -U vllm openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.9/43.9 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.0/316.0 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 97.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.7/322.7 kB 29.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 116.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 782.6/782.6 kB 53.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 99.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.2

In [ ]:
import os
os.kill(os.getpid(), 9)

## Policy Design

### Problem

The baseline experiments showed that long-prefill requests can increase the TTFT of short requests that arrive shortly afterward.

In the default scheduler configuration, a long prefill may consume a large portion of the available token budget in one scheduler iteration.

For example:

Long request arrives  
↓  
Long prefill requires more than 2000 tokens  
↓  
Scheduler allows a large prefill chunk  
↓  
Short request arrives while the long prefill is executing  
↓  
Short request waits for the next scheduling opportunity  
↓  
TTFT increases  

The baseline streaming experiment showed that reducing `max_num_batched_tokens` from 8192 to 512 reduced short-request TTFT while TPOT remained nearly unchanged.

This suggests that limiting long-prefill work can improve scheduler responsiveness.

---

### Design Goal

The goal is to reduce interference from long prefills without globally reducing the scheduler token budget.

Instead of changing:

`max_num_batched_tokens`

for every workload, the scheduler will selectively limit the number of tokens assigned to long-prefill requests.

The global scheduler budget can therefore remain large while individual long-prefill requests are prevented from consuming too much of it in a single scheduler iteration.

---

### Initial Policy

The first policy introduces a maximum per-request prefill chunk size.

For a request that is still in the prefill phase:

`scheduled_prefill_tokens = min(num_new_tokens, PREFILL_CHUNK_CAP)`

For decode requests, the scheduler behavior remains unchanged.

For short prefills that are already smaller than the cap, the scheduler behavior also remains unchanged.

Example with:

`PREFILL_CHUNK_CAP = 512`

A 2452-token prompt would be scheduled as:

512  
↓  
512  
↓  
512  
↓  
512  
↓  
404  

instead of potentially scheduling the full prefill in one iteration.

---

### Policy Scope

The policy applies only to requests that are still processing prompt tokens.

Conceptually:

If request is in prefill:

`num_computed_tokens < num_prompt_tokens`

then:

`num_new_tokens = min(num_new_tokens, PREFILL_CHUNK_CAP)`

Otherwise:

decode scheduling remains unchanged.

This keeps the optimization focused on prefill interference.

---

### What the Policy Does Not Change

The policy should not modify:

- request priority
- waiting queue order
- running queue order
- KV-cache allocation logic
- preemption policy
- decode scheduling
- model execution
- prefix caching behavior
- sampling behavior

Only the number of prefill tokens scheduled for a request in one scheduler iteration is changed.

This makes the experiment easier to reason about and isolates the effect of prefill chunk size.

---

### Why Not Only Reduce the Global Token Budget?

Reducing `max_num_batched_tokens` from 8192 to 512 already improved short-request TTFT in the baseline experiment.

However, this changes the scheduling capacity of the entire system.

A global 512-token budget also limits workloads that may not need such a restriction.

The proposed policy instead keeps:

`global token budget = 8192`

while applying:

`long-prefill per-request cap = 512`

Conceptually:

Global budget: 8192 tokens

Long prefill A:
maximum 512 tokens

Long prefill B:
maximum 512 tokens

Short request:
can use remaining scheduler capacity

Decode requests:
continue to receive their normal token allocations

This allows the scheduler to retain a large global scheduling capacity while preventing a single long prefill from dominating one iteration.

---

### Expected Scheduler Behavior

Without the policy:

Long prefill  
↓  
large token allocation  
↓  
large portion of scheduler budget consumed  
↓  
fewer scheduling opportunities for newly arrived requests  

With the policy:

Long prefill  
↓  
prefill chunk capped  
↓  
remaining scheduler capacity preserved  
↓  
other requests can be admitted or scheduled  
↓  
short-request TTFT may decrease  

---

### Expected Benefits

The policy may improve:

- short-request TTFT
- p95 TTFT
- mixed-workload responsiveness
- fairness between long and short requests

The policy should have limited impact on:

- decode TPOT
- short-prompt workloads
- decode-heavy workloads

---

### Possible Trade-offs

Smaller prefill chunks may also introduce costs.

A long prompt may require more scheduler iterations:

Large chunk:

2452 tokens  
↓  
1 scheduling decision  

Small chunks:

512 + 512 + 512 + 512 + 404  
↓  
5 scheduling decisions  

This may increase:

- scheduler overhead
- number of model execution iterations
- kernel launch overhead
- long-request TTFT
- total prefill completion time

Therefore, the smallest chunk size is not necessarily the best policy.

The optimization is a latency-throughput trade-off.

---

### Initial Hypothesis

The main hypothesis is:

A moderate per-request prefill cap can reduce short-request TTFT under mixed workloads without significantly degrading TPOT or total throughput.

The expected behavior is:

Smaller prefill chunk  
↓  
more frequent scheduler opportunities  
↓  
lower short-request TTFT  

while:

Decode execution  
↓  
mostly unchanged  
↓  
similar TPOT  

---

### First Experimental Policy

The first implementation will use:

`PREFILL_CHUNK_CAP = 512`

while keeping:

`max_num_batched_tokens = 8192`

This is intentionally chosen because the baseline experiment already showed that a 512-token global budget improved short-request TTFT.

The new experiment will test whether similar latency improvements can be achieved using a per-request prefill cap while preserving the larger global scheduler budget.

---

### Later Ablation

After validating the implementation, the policy will be evaluated with several chunk sizes:

- no custom cap
- 1024 tokens
- 512 tokens
- 256 tokens

The goal is to identify the trade-off between:

- short-request TTFT
- long-request TTFT
- TPOT
- end-to-end latency
- throughput
- scheduler overhead

The expected relationship is:

Large cap  
→ fewer scheduler iterations  
→ better long-prefill efficiency  
→ potentially worse short-request responsiveness  

Small cap  
→ more scheduler opportunities  
→ potentially better short-request TTFT  
→ potentially higher scheduling overhead  

The best configuration is expected to lie between these extremes.

## Locate the Modification Point

The custom policy should modify only the number of prefill tokens scheduled for an individual request.

The safest insertion point is after `num_new_tokens` has been calculated, but before KV-cache allocation and scheduler accounting.

The desired order is:

1. compute the request's remaining work
2. apply existing scheduler limits
3. apply the custom per-request prefill cap
4. allocate KV-cache slots
5. record the scheduled tokens
6. update the global token budget

The policy should not be inserted after KV allocation because the memory allocation would already have been calculated using the old token count.

The policy should also not replace the existing token-budget logic. It should act as an additional per-request constraint.

In [1]:
# =============================================================================
# Inspect Scheduler.schedule()
# =============================================================================

import inspect
from vllm.v1.core.sched.scheduler import Scheduler

schedule_source = inspect.getsource(Scheduler.schedule)

print(schedule_source)

    def schedule(self, throttle_prefills: bool = False) -> SchedulerOutput:
        self.current_step += 1
        # NOTE(woosuk) on the scheduling algorithm:
        # There's no "decoding phase" nor "prefill phase" in the scheduler.
        # Each request just has the num_computed_tokens and
        # num_tokens_with_spec. num_tokens_with_spec =
        # len(prompt_token_ids) + len(output_token_ids) + len(spec_token_ids).
        # At each step, the scheduler tries to assign tokens to the requests
        # so that each request's num_computed_tokens can catch up its
        # num_tokens_with_spec. This is general enough to cover
        # chunked prefills, prefix caching, speculative decoding,
        # and the "jump decoding" optimization in the future.

        scheduled_new_reqs: list[Request] = []
        scheduled_resumed_reqs: list[Request] = []
        scheduled_running_reqs: list[Request] = []
        preempted_reqs: list[Request] = []

        req_to_new_blocks: dict[str, K

## Implement the Adaptive Prefill Policy

The custom scheduler policy is implemented as a small helper so that both the RUNNING and WAITING scheduling paths use the same logic.

The policy keeps the global scheduler budget unchanged while selectively capping prefill work when responsiveness matters.

The initial rule is:

- cap the first prefill chunk to 512 tokens
- continue using the 512-token cap when there is scheduling contention
- allow larger prefill chunks when no competing requests are present
- never cap decode work

This preserves the large global scheduler budget while reducing the chance that one long prefill monopolizes a scheduling iteration.

In [2]:
# Conceptual helper

def _apply_adaptive_prefill_cap(
    self,
    request,
    num_computed_tokens,
    num_new_tokens,
):
    PREFILL_CAP = 512

    is_prefill = (
        num_computed_tokens
        < request.num_prompt_tokens
    )

    if not is_prefill:
        return num_new_tokens

    is_first_prefill_chunk = (
        num_computed_tokens == 0
    )

    has_waiting_requests = bool(
        self.waiting or self.skipped_waiting
    )

    has_decode_competition = any(
        other is not request
        and other.num_computed_tokens >= other.num_prompt_tokens
        for other in self.running
    )

    has_contention = (
        has_waiting_requests
        or has_decode_competition
    )

    if is_first_prefill_chunk or has_contention:
        num_new_tokens = min(
            num_new_tokens,
            PREFILL_CAP,
        )

    return num_new_tokens

In [3]:
# =============================================================================
# Locate scheduler.py
# =============================================================================

import inspect
from pathlib import Path
from vllm.v1.core.sched.scheduler import Scheduler

scheduler_path = Path(inspect.getfile(Scheduler))

print(scheduler_path)

/usr/local/lib/python3.13/dist-packages/vllm/v1/core/sched/scheduler.py


In [4]:
# =============================================================================
# Backup scheduler.py
# =============================================================================

backup_path = scheduler_path.with_suffix(".py.adaptive_prefill_backup")

if not backup_path.exists():
    backup_path.write_text(scheduler_path.read_text())
    print("Backup created:", backup_path)
else:
    print("Backup already exists:", backup_path)

Backup created: /usr/local/lib/python3.13/dist-packages/vllm/v1/core/sched/scheduler.py.adaptive_prefill_backup


In [5]:
# =============================================================================
# Insert adaptive prefill helper
# =============================================================================

source = scheduler_path.read_text()

marker = "    def schedule(self, throttle_prefills: bool = False) -> SchedulerOutput:\n"

helper = '''
    def _apply_adaptive_prefill_cap(
        self,
        request,
        num_computed_tokens: int,
        num_new_tokens: int,
    ) -> int:
        """Limit prefill chunk size when responsiveness matters."""
        prefill_cap = 512

        is_prefill = num_computed_tokens < request.num_prompt_tokens
        if not is_prefill:
            return num_new_tokens

        is_first_prefill_chunk = num_computed_tokens == 0

        has_waiting_requests = bool(
            self.waiting or self.skipped_waiting
        )

        has_decode_competition = any(
            other is not request
            and other.num_computed_tokens >= other.num_prompt_tokens
            for other in self.running
        )

        has_contention = (
            has_waiting_requests
            or has_decode_competition
        )

        if is_first_prefill_chunk or has_contention:
            num_new_tokens = min(
                num_new_tokens,
                prefill_cap,
            )

        return num_new_tokens

'''

if "_apply_adaptive_prefill_cap" not in source:
    if marker not in source:
        raise RuntimeError("Could not find schedule() insertion point.")

    source = source.replace(
        marker,
        helper + marker,
        1,
    )

    scheduler_path.write_text(source)
    print("Adaptive prefill helper inserted.")
else:
    print("Helper already exists.")

Adaptive prefill helper inserted.


## RUNNING path

In [6]:
# =============================================================================
# Patch RUNNING path
# =============================================================================

source = scheduler_path.read_text()

old = '''            num_new_tokens = self._reserve_prefill_lookahead(
                request, request.num_computed_tokens, num_new_tokens
            )

            if num_new_tokens == 0:
'''

new = '''            num_new_tokens = self._reserve_prefill_lookahead(
                request, request.num_computed_tokens, num_new_tokens
            )

            num_new_tokens = self._apply_adaptive_prefill_cap(
                request,
                request.num_computed_tokens,
                num_new_tokens,
            )

            if num_new_tokens == 0:
'''

if old in source:
    source = source.replace(old, new, 1)
    scheduler_path.write_text(source)
    print("RUNNING path patched.")
elif "_apply_adaptive_prefill_cap(" in source:
    print("RUNNING path may already be patched.")
else:
    raise RuntimeError("RUNNING insertion point not found.")

RUNNING path patched.


## WAITING path

In [7]:
# =============================================================================
# Patch WAITING path
# =============================================================================

source = scheduler_path.read_text()

old = '''                    num_new_tokens = self._reserve_prefill_lookahead(
                        request, num_computed_tokens, num_new_tokens
                    )

                    if num_new_tokens == 0:
'''

new = '''                    num_new_tokens = self._reserve_prefill_lookahead(
                        request, num_computed_tokens, num_new_tokens
                    )

                    num_new_tokens = self._apply_adaptive_prefill_cap(
                        request,
                        num_computed_tokens,
                        num_new_tokens,
                    )

                    if num_new_tokens == 0:
'''

if old in source:
    source = source.replace(old, new, 1)
    scheduler_path.write_text(source)
    print("WAITING path patched.")
else:
    print("WAITING insertion point not found or already patched.")

WAITING path patched.


In [8]:
# =============================================================================
# Syntax validation
# =============================================================================

import py_compile

py_compile.compile(
    str(scheduler_path),
    doraise=True,
)

print("scheduler.py syntax OK")

scheduler.py syntax OK


In [9]:
source = scheduler_path.read_text()

print(
    "helper count:",
    source.count("def _apply_adaptive_prefill_cap")
)

print(
    "call count:",
    source.count("self._apply_adaptive_prefill_cap(")
)

helper count: 1
call count: 2


## Adaptive Prefill Policy Validation

Before running performance benchmarks, the modified scheduler is validated with a small controlled workload.

The goal is to confirm that the custom policy changes scheduling behavior as intended.

The validation checks:

- the global scheduler token budget remains 8192
- the first long-prefill chunk is capped at 512 tokens
- later prefill chunks can use a larger allocation when there is no contention
- long-prefill work is capped again when competing requests are present
- decode requests are not capped by the custom policy

A lightweight scheduler trace is used to observe the runtime decisions directly.

In [10]:
# =============================================================================
# Add lightweight scheduler tracer
# =============================================================================

source = scheduler_path.read_text()

# Add imports if needed.
if "import json\n" not in source:
    source = source.replace(
        "import ",
        "import json\nimport os\nimport ",
        1,
    )

# Add trace state after current_step initialization.
old = "        self.current_step = 0\n"

new = """        self.current_step = 0

        self._adaptive_trace_enabled = (
            os.environ.get("VLLM_ADAPTIVE_TRACE", "0") == "1"
        )
        self._adaptive_trace_path = os.environ.get(
            "VLLM_ADAPTIVE_TRACE_PATH",
            "/tmp/vllm_adaptive_trace.jsonl",
        )
"""

if old in source and "_adaptive_trace_enabled" not in source:
    source = source.replace(old, new, 1)

scheduler_path.write_text(source)

147397

In [11]:
# =============================================================================
# Insert trace helper
# =============================================================================

source = scheduler_path.read_text()

marker = "    def _apply_adaptive_prefill_cap(\n"

trace_helper = '''
    def _adaptive_trace(self, event: str, **fields) -> None:
        if not self._adaptive_trace_enabled:
            return

        record = {
            "step": self.current_step,
            "event": event,
            **fields,
        }

        with open(self._adaptive_trace_path, "a") as f:
            f.write(json.dumps(record) + "\\n")

'''

if "def _adaptive_trace(" not in source:
    if marker not in source:
        raise RuntimeError("Could not find adaptive helper.")

    source = source.replace(
        marker,
        trace_helper + marker,
        1,
    )

    scheduler_path.write_text(source)

print("Trace helper inserted.")

Trace helper inserted.


In [12]:
# =============================================================================
# Add policy-decision tracing
# =============================================================================

source = scheduler_path.read_text()

start = source.index("    def _apply_adaptive_prefill_cap(")
end = source.index("\n    def schedule(", start)

new_helper = '''    def _apply_adaptive_prefill_cap(
        self,
        request,
        num_computed_tokens: int,
        num_new_tokens: int,
    ) -> int:
        """Limit prefill chunk size when responsiveness matters."""
        prefill_cap = 512

        original_num_new_tokens = num_new_tokens

        is_prefill = (
            num_computed_tokens
            < request.num_prompt_tokens
        )

        if not is_prefill:
            self._adaptive_trace(
                "policy_decision",
                request_id=request.request_id,
                phase="decode",
                num_computed_tokens=num_computed_tokens,
                original_tokens=original_num_new_tokens,
                final_tokens=num_new_tokens,
                capped=False,
                reason="decode",
            )
            return num_new_tokens

        is_first_prefill_chunk = (
            num_computed_tokens == 0
        )

        has_waiting_requests = bool(
            self.waiting or self.skipped_waiting
        )

        has_decode_competition = any(
            other is not request
            and other.num_computed_tokens >= other.num_prompt_tokens
            for other in self.running
        )

        has_contention = (
            has_waiting_requests
            or has_decode_competition
        )

        reason = "none"

        if is_first_prefill_chunk:
            reason = "first_prefill"
        elif has_contention:
            reason = "contention"

        if is_first_prefill_chunk or has_contention:
            num_new_tokens = min(
                num_new_tokens,
                prefill_cap,
            )

        self._adaptive_trace(
            "policy_decision",
            request_id=request.request_id,
            phase="prefill",
            num_computed_tokens=num_computed_tokens,
            original_tokens=original_num_new_tokens,
            final_tokens=num_new_tokens,
            capped=num_new_tokens < original_num_new_tokens,
            reason=reason,
            has_waiting_requests=has_waiting_requests,
            has_decode_competition=has_decode_competition,
        )

        return num_new_tokens

'''

source = source[:start] + new_helper + source[end:]
scheduler_path.write_text(source)

print("Adaptive helper updated with tracing.")

Adaptive helper updated with tracing.


In [13]:
import py_compile

py_compile.compile(
    str(scheduler_path),
    doraise=True,
)

print("scheduler.py syntax OK")

scheduler.py syntax OK


In [14]:
%%writefile /content/test_adaptive_policy.py

from vllm import LLM, SamplingParams

llm = LLM(
    model="Qwen/Qwen2.5-1.5B-Instruct",
    max_model_len=4096,
    gpu_memory_utilization=0.70,
    dtype="float16",
    max_num_batched_tokens=8192,
    enable_chunked_prefill=True,
)

sampling = SamplingParams(
    temperature=0.0,
    max_tokens=32,
)

long_prompt = "Explain GPU memory hierarchy in detail. " * 350

prompts = [
    long_prompt,
    "What is a GPU warp?",
    "What is KV cache?",
]

outputs = llm.generate(
    prompts,
    sampling,
    use_tqdm=False,
)

for output in outputs:
    print(
        output.request_id,
        len(output.prompt_token_ids),
        len(output.outputs[0].token_ids),
    )

Writing /content/test_adaptive_policy.py


In [15]:
from pathlib import Path

trace_path = Path("/content/adaptive_trace.jsonl")

if trace_path.exists():
    trace_path.unlink()

In [16]:
!VLLM_ADAPTIVE_TRACE=1 \
VLLM_ADAPTIVE_TRACE_PATH=/content/adaptive_trace.jsonl \
python /content/test_adaptive_policy.py

INFO 09-21 02:34:48 [api_utils.py:286] non-default args: {'dtype': 'float16', 'max_model_len': 4096, 'gpu_memory_utilization': 0.7, 'max_num_batched_tokens': 8192, 'disable_log_stats': True, 'enable_chunked_prefill': True, 'model': 'Qwen/Qwen2.5-1.5B-Instruct'}
WARNING 09-21 02:34:48 [envs.py:2248] Unknown vLLM environment variable detected: VLLM_ADAPTIVE_TRACE
WARNING 09-21 02:34:48 [envs.py:2248] Unknown vLLM environment variable detected: VLLM_ADAPTIVE_TRACE_PATH
config.json: 100% 660/660 [00:00<00:00, 3.43MB/s]
INFO 09-21 02:35:08 [model.py:684] Resolved architecture: Qwen2ForCausalLM
WARNING 09-21 02:35:08 [model.py:2355] Casting torch.bfloat16 to torch.float16.
INFO 09-21 02:35:08 [model.py:2021] Using max model len 4096
INFO 09-21 02:35:08 [scheduler.py:277] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 09-21 02:35:08 [kernel.py:369] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])
to

In [17]:
import json
from pathlib import Path

records = [
    json.loads(line)
    for line in Path("/content/adaptive_trace.jsonl").read_text().splitlines()
]

for r in records[:30]:
    print(
        f"step={r['step']:2d} | "
        f"req={r['request_id'][:10]:10s} | "
        f"phase={r['phase']:7s} | "
        f"computed={r['num_computed_tokens']:4d} | "
        f"original={r['original_tokens']:4d} | "
        f"final={r['final_tokens']:4d} | "
        f"capped={str(r['capped']):5s} | "
        f"reason={r['reason']}"
    )

step= 1 | req=0-a326cf45 | phase=prefill | computed=   0 | original=2452 | final= 512 | capped=True  | reason=first_prefill
step= 2 | req=0-a326cf45 | phase=prefill | computed= 512 | original=1940 | final= 512 | capped=True  | reason=contention
step= 2 | req=1-93200fb5 | phase=prefill | computed=   0 | original=   6 | final=   6 | capped=False | reason=first_prefill
step= 2 | req=2-92bc78f9 | phase=prefill | computed=   0 | original=   5 | final=   5 | capped=False | reason=first_prefill
step= 3 | req=0-a326cf45 | phase=prefill | computed=1024 | original=1428 | final= 512 | capped=True  | reason=contention
step= 3 | req=1-93200fb5 | phase=decode  | computed=   6 | original=   1 | final=   1 | capped=False | reason=decode
step= 3 | req=2-92bc78f9 | phase=decode  | computed=   5 | original=   1 | final=   1 | capped=False | reason=decode
step= 4 | req=0-a326cf45 | phase=prefill | computed=1536 | original= 916 | final= 512 | capped=True  | reason=contention
step= 4 | req=1-93200fb5 | phas

In [19]:
%%writefile /content/test_adaptive_no_contention.py

from vllm import LLM, SamplingParams

llm = LLM(
    model="Qwen/Qwen2.5-1.5B-Instruct",
    max_model_len=4096,
    gpu_memory_utilization=0.70,
    dtype="float16",
    max_num_batched_tokens=8192,
    enable_chunked_prefill=True,
)

sampling = SamplingParams(
    temperature=0.0,
    max_tokens=32,
)

long_prompt = "Explain GPU memory hierarchy in detail. " * 350

outputs = llm.generate(
    [long_prompt],
    sampling,
    use_tqdm=False,
)

for output in outputs:
    print(
        output.request_id,
        len(output.prompt_token_ids),
        len(output.outputs[0].token_ids),
    )

Overwriting /content/test_adaptive_no_contention.py


In [20]:
from pathlib import Path

trace_path = Path("/content/adaptive_trace.jsonl")

if trace_path.exists():
    trace_path.unlink()

In [21]:
!VLLM_ADAPTIVE_TRACE=1 \
VLLM_ADAPTIVE_TRACE_PATH=/content/adaptive_trace.jsonl \
python /content/test_adaptive_no_contention.py

INFO 09-21 02:38:47 [api_utils.py:286] non-default args: {'dtype': 'float16', 'max_model_len': 4096, 'gpu_memory_utilization': 0.7, 'max_num_batched_tokens': 8192, 'disable_log_stats': True, 'enable_chunked_prefill': True, 'model': 'Qwen/Qwen2.5-1.5B-Instruct'}
WARNING 09-21 02:38:47 [envs.py:2248] Unknown vLLM environment variable detected: VLLM_ADAPTIVE_TRACE
WARNING 09-21 02:38:47 [envs.py:2248] Unknown vLLM environment variable detected: VLLM_ADAPTIVE_TRACE_PATH
INFO 09-21 02:38:48 [model.py:684] Resolved architecture: Qwen2ForCausalLM
WARNING 09-21 02:38:48 [model.py:2355] Casting torch.bfloat16 to torch.float16.
INFO 09-21 02:38:48 [model.py:2021] Using max model len 4096
INFO 09-21 02:38:48 [scheduler.py:277] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 09-21 02:38:48 [kernel.py:369] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])
(EngineCore pid=6897) INFO 09-21 02:38:53 [core.py:1

In [22]:
import json
from pathlib import Path

records = [
    json.loads(line)
    for line in Path("/content/adaptive_trace.jsonl").read_text().splitlines()
]

for r in records[:30]:
    print(
        f"step={r['step']:2d} | "
        f"req={r['request_id'][:10]:10s} | "
        f"phase={r['phase']:7s} | "
        f"computed={r['num_computed_tokens']:4d} | "
        f"original={r['original_tokens']:4d} | "
        f"final={r['final_tokens']:4d} | "
        f"capped={str(r['capped']):5s} | "
        f"reason={r['reason']}"
    )

step= 1 | req=0-90cf11d8 | phase=prefill | computed=   0 | original=2452 | final= 512 | capped=True  | reason=first_prefill
step= 2 | req=0-90cf11d8 | phase=prefill | computed= 512 | original=1940 | final=1940 | capped=False | reason=none
step= 3 | req=0-90cf11d8 | phase=decode  | computed=2452 | original=   1 | final=   1 | capped=False | reason=decode
step= 4 | req=0-90cf11d8 | phase=decode  | computed=2453 | original=   1 | final=   1 | capped=False | reason=decode
step= 5 | req=0-90cf11d8 | phase=decode  | computed=2454 | original=   1 | final=   1 | capped=False | reason=decode
step= 6 | req=0-90cf11d8 | phase=decode  | computed=2455 | original=   1 | final=   1 | capped=False | reason=decode
step= 7 | req=0-90cf11d8 | phase=decode  | computed=2456 | original=   1 | final=   1 | capped=False | reason=decode
step= 8 | req=0-90cf11d8 | phase=decode  | computed=2457 | original=   1 | final=   1 | capped=False | reason=decode
step= 9 | req=0-90cf11d8 | phase=decode  | computed=2458 | 